In [1]:
import numpy as np
import pandas as pd
from copy import copy
from scipy.stats import norm
from scipy.optimize import brentq

# FINM 32000 HW2

In [2]:
class CallOption:
    def __init__(self, K, T, price=None):
        self.K = K
        self.T = T
        self.price = price


class GBMdynamics:
    def __init__(self, S=None, X=None, r=0.0, rGrow=0.0, sigma=None):
        if S is None and X is None:
            raise ValueError("Provide S or X.")
        self.S = X if S is None else S
        self.X = self.S
        self.r = r
        self.rGrow = rGrow
        self.sigma = sigma

    def update_sigma(self, sigma):
        self.sigma = sigma
        return self


class AnalyticEngine:
    def BSpriceCall(self, dynamics, contract):
        F = dynamics.S * np.exp(dynamics.rGrow * contract.T)
        std = dynamics.sigma * np.sqrt(contract.T)
        d1 = np.log(F / contract.K) / std + std / 2
        d2 = d1 - std
        return np.exp(-dynamics.r * contract.T) * (F * norm.cdf(d1) - contract.K * norm.cdf(d2))

    def BSpricePut(self, dynamics, contract):
        F = dynamics.S * np.exp(dynamics.rGrow * contract.T)
        std = dynamics.sigma * np.sqrt(contract.T)
        d1 = np.log(F / contract.K) / std + std / 2
        d2 = d1 - std
        return np.exp(-dynamics.r * contract.T) * (contract.K * norm.cdf(-d2) - F * norm.cdf(-d1))

    def IV(self, dynamics, contract):
        if contract.price is None:
            raise ValueError("Contract price must be given.")

        df = np.exp(-dynamics.r * contract.T)
        F = dynamics.S * np.exp(dynamics.rGrow * contract.T)
        lowerbound = max(0.0, df * (F - contract.K))
        C = contract.price

        if C < lowerbound or C >= df * F:
            return np.nan
        if C == lowerbound:
            return 0.0

        dynamics_try = copy(dynamics)

        sigma_try = 0.2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) > C:
            sigma_try /= 2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) < C:
            sigma_try *= 2

        hi = sigma_try
        lo = hi / 2

        return brentq(
            lambda sigma: self.BSpriceCall(dynamics_try.update_sigma(sigma), contract) - C,
            lo,
            hi
        )

## Problem 1

In [3]:
class UpAndOutPut:
    def __init__(self, K, T, barrier, observationinterval):
        self.K = K
        self.T = T
        self.barrier = barrier
        self.observationinterval = observationinterval


hw2contract = UpAndOutPut(K=95, T=0.25, barrier=114, observationinterval=0.02)
hw2dynamics = GBMdynamics(S=100, sigma=0.4, rGrow=0, r=0)

In [4]:
class TreeEngine:
    def __init__(self, N):
        self.N = N

    def price_upandout(self, dynamics, contract):
        deltat = contract.T / self.N

        numTimestepsPerObs = contract.observationinterval / deltat
        if abs(numTimestepsPerObs - round(numTimestepsPerObs)) > 1e-8:
            raise ValueError("This value of N fails to place the observation dates in the tree.")
        numTimestepsPerObs = int(round(numTimestepsPerObs))

        J = int(np.ceil(np.log(contract.barrier / dynamics.S) / (dynamics.sigma * np.sqrt(3 * deltat)) - 0.5))
        deltax = np.log(contract.barrier / dynamics.S) / (J + 0.5)

        nu = dynamics.rGrow - 0.5 * dynamics.sigma**2
        Pu = 0.5 * (((dynamics.sigma**2) * deltat + (nu**2) * (deltat**2)) / (deltax**2) + nu * deltat / deltax)
        Pd = 0.5 * (((dynamics.sigma**2) * deltat + (nu**2) * (deltat**2)) / (deltax**2) - nu * deltat / deltax)
        Pm = 1.0 - ((dynamics.sigma**2) * deltat + (nu**2) * (deltat**2)) / (deltax**2)

        j = np.arange(self.N, -self.N - 1, -1)
        Sgrid = dynamics.S * np.exp(j * deltax)

        optionprice = np.maximum(contract.K - Sgrid, 0.0)
        disc = np.exp(-dynamics.r * deltat)

        for n in range(self.N - 1, -1, -1):
            optionprice = disc * (Pu * optionprice[:-2] + Pm * optionprice[1:-1] + Pd * optionprice[2:])

            if n > 0 and n % numTimestepsPerObs == 0 and n * deltat < contract.T - 1e-12:
                j_now = np.arange(n, -n - 1, -1)
                S_now = dynamics.S * np.exp(j_now * deltax)
                optionprice = np.where(S_now >= contract.barrier, 0.0, optionprice)

        return optionprice[0]

In [5]:
hw2tree = TreeEngine(N=100)
hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.311979858729038)

In [6]:
N_list = [25, 50, 75, 100, 125, 150, 175, 200, 250, 300, 400, 500, 600, 800]
prices = [TreeEngine(N).price_upandout(hw2dynamics, hw2contract) for N in N_list]

conv = pd.DataFrame({"N": N_list, "price": prices})
conv["abs_change"] = conv["price"].diff().abs()
conv

,N,price,abs_change
0,25,5.308850,NaN
1,50,5.327181,0.018331
2,75,5.322643,0.004539
3,100,5.311980,0.010663
4,125,5.310649,0.001331
5,150,5.306095,0.004554
6,175,5.305464,0.000631
7,200,5.304992,0.000472
8,250,5.307893,0.002901
9,300,5.301942,0.005951


In [7]:
N_star = 400
price_uo = TreeEngine(N_star).price_upandout(hw2dynamics, hw2contract)
price_uo

np.float64(5.304649871106281)

For part (a), using $N=400$ gives a converged price in my judgment:
$$
V_{\text{UO, disc}}(0) \approx 5.304650.
$$

Why place $\log H$ halfway between adjacent log-price levels? Because the barrier creates a discontinuity in value across nearby nodes. Putting the barrier halfway between two levels makes the live/knocked-out classification locally symmetric, which reduces spatial bias and improves convergence relative to forcing the barrier exactly onto a node.

In [8]:
analytic = AnalyticEngine()
plain_put = analytic.BSpricePut(hw2dynamics, CallOption(K=95, T=0.25))
price_ui = plain_put - price_uo

plain_put, price_ui

(np.float64(5.519541063676975), np.float64(0.21489119257069422))

For part (b), up-and-in plus up-and-out equals the plain vanilla put, so
$$
V_{\text{UI, disc}}(0)=V_{\text{plain put}}(0)-V_{\text{UO, disc}}(0).
$$
Using the tree price above,
$$
V_{\text{UI, disc}}(0) \approx 0.214891.
$$

For part (c1), the continuously monitored up-and-out put is **smaller** than the discretely monitored one, because continuous monitoring knocks the option out on a larger set of paths.

In [9]:
H = hw2contract.barrier
K = hw2contract.K
K_mirror = H**2 / K
alpha = K / H

mirror_call = analytic.BSpriceCall(hw2dynamics, CallOption(K=K_mirror, T=hw2contract.T))
price_uo_cont = plain_put - alpha * mirror_call

alpha, K_mirror, mirror_call, price_uo_cont

(0.8333333333333334,
 136.8,
 np.float64(0.5856175597234898),
 np.float64(5.031526430574067))

For part (c2), at a barrier hit time $\tau$ we have $S_\tau = H = 114$. Let the remaining maturity be $u=T-\tau$. Then
$$
P(H,K,u)=K N(-d_2)-H N(-d_1),
$$
while for the call with strike
$$
K'=\frac{H^2}{K}=\frac{114^2}{95}=136.8,
$$
we have
$$
C(H,K',u)=H N(-d_2)-K' N(-d_1).
$$
Hence
$$
P(H,K,u)=\frac{K}{H} C(H,K',u).
$$
So the required hedge ratio is
$$
\alpha=\frac{K}{H}=\frac{95}{114}=\frac{5}{6}.
$$

Therefore the continuously monitored up-and-out put is replicated by
$$
1\times \text{put}(K=95)-\frac{95}{114}\times \text{call}(K=136.8),
$$
and its time-0 value is
$$
V_{\text{UO, cont}}(0)=P(100,95,0.25)-\frac{95}{114}C(100,136.8,0.25)\approx 5.031526.
$$

## Problem 2

In [10]:
hw2analytic = AnalyticEngine()

iv_05 = hw2analytic.IV(
    GBMdynamics(S=100, sigma=None, rGrow=0, r=0),
    CallOption(K=100, T=0.5, price=11.25)
)

iv_10 = hw2analytic.IV(
    GBMdynamics(S=100, sigma=None, rGrow=0, r=0),
    CallOption(K=100, T=1.0, price=12.00)
)

iv_mid = 0.5 * (iv_05 + iv_10)

price_075_midiv = hw2analytic.BSpriceCall(
    GBMdynamics(S=100, sigma=iv_mid, rGrow=0, r=0),
    CallOption(K=100, T=0.75)
)

pd.DataFrame({
    "expiry": [0.5, 1.0, 0.75],
    "implied_vol": [iv_05, iv_10, iv_mid],
    "call_price": [11.25, 12.00, price_075_midiv]
})

,expiry,implied_vol,call_price
0,0.50,0.400133,11.250000
1,1.00,0.301938,12.000000
2,0.75,0.351036,12.081533


Thus
$$
\sigma_{\text{imp}}(0.5) \approx 0.400133, \qquad
\sigma_{\text{imp}}(1.0) \approx 0.301938.
$$
Using the midpoint implied volatility,
$$
\sigma_{\text{mid}}=\frac{0.400133+0.301938}{2}\approx 0.351036,
$$
the 0.75-year call price is
$$
C(100,0.75) \approx 12.081533.
$$

Assume **cash settlement**.

This price admits calendar-spread arbitrage because it is larger than the 1-year call price, even though the 1-year call has the same strike and strictly longer maturity.

A valid arbitrage is:

1. Short the 0.75-year call for $12.081533$.
2. Long the 1-year call for $12.00$.
3. Receive $0.081533$ at time 0.

At $t=0.75$:
- If $S_{0.75} \le 100$, the short call expires worthless, while the long 1-year call still has nonnegative value.
- If $S_{0.75} > 100$, the short call pays $S_{0.75}-100$, but the remaining 0.25-year call with the same strike is worth at least its intrinsic value $S_{0.75}-100$, so selling the long 1-year call covers the liability.

Hence the strategy has positive initial cashflow and never produces a negative future cashflow.